# A bigram baseline with sentence boundaries

Before training a GPT on short image captions, a team wants the score of the simplest
language model, a bigram model, which predicts each word from the word before it alone.
Each caption is a separate text: it starts after a start token `<s>`, and it ends with an
end token `</s>` that the model must predict too, since that is how a model learns when to
stop. You will count word pairs, smooth the counts, compute perplexity, and choose the
smoothing on validation captions.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# Captions from a toy grammar: a subject, a verb, an object and sometimes an ending.
SUBJECTS = ["the robot", "a child", "my sister", "the old dog", "our teacher"]
OBJECTS = {
    "reads": ["a map", "two books", "the sign"],
    "draws": ["a map", "a red kite", "the sign", "two plums"],
    "eats": ["the apple", "two plums"],
    "finds": ["a red kite", "two books", "the apple", "a map"],
}
ENDINGS = ["", "today", "again", "at noon", "quietly"]


def caption(rng):
    verb = list(OBJECTS)[rng.choice(4, p=[0.4, 0.3, 0.2, 0.1])]
    parts = [SUBJECTS[rng.integers(5)], verb, OBJECTS[verb][rng.integers(len(OBJECTS[verb]))],
             ENDINGS[rng.choice(5, p=[0.62, 0.16, 0.12, 0.07, 0.03])]]
    return " ".join(parts).split()


rng = np.random.default_rng(2026)
train = [caption(rng) for _ in range(60)]
valid = [caption(rng) for _ in range(200)]
test = [caption(rng) for _ in range(200)]

vocab = ["<s>", "</s>"] + sorted({w for s in train for w in s})
index = {w: i for i, w in enumerate(vocab)}
START, END = index["<s>"], index["</s>"]
print(f"{len(vocab)} tokens. First captions: {' '.join(train[0])} / {' '.join(train[1])}")

## Part 1 (coding task): Counting pairs

Write `count_pairs(sentences, index)`. Each sentence is a list of words. Put `<s>` before
each sentence and `</s>` after it, then count the pairs of neighbors: entry `[i, j]` of
the result is the number of times token `j` comes right after token `i`. Each sentence is
a separate text. Return a $V \times V$ array of integers, where $V$ is `len(index)`.

In [ ]:
def count_pairs(sentences, index):
    """V x V counts: entry [i, j] is how often token j comes right after token i."""
    # TODO
    raise NotImplementedError

In [ ]:
toy = {"<s>": 0, "</s>": 1, "a": 2, "b": 3}
C_toy = np.asarray(count_pairs([["a", "a", "a"], ["b"]], toy))
assert C_toy.shape == (4, 4), f"expected a 4 x 4 array for 4 tokens, got shape {C_toy.shape}"
assert np.issubdtype(C_toy.dtype, np.integer), "the counts should be integers"
assert C_toy[1, 0] == 0 and C_toy[2, 3] == 0, "a pair runs from one sentence into the next: count each sentence on its own"
assert C_toy[2, 2] == 2, f"the pair (a, a) occurs twice in the first sentence, got {C_toy[2, 2]} (C[i, j] += 1 with arrays of indices counts a repeated pair once)"
assert C_toy.sum() == 6, f"sentences of 3 words and 1 word give 4 + 2 = 6 pairs, one per word and one for </s>; got {C_toy.sum()}"
expected = np.zeros((4, 4), dtype=int)
expected[0, 2], expected[2, 2], expected[2, 1], expected[0, 3], expected[3, 1] = 1, 2, 1, 1, 1
assert np.array_equal(C_toy, expected), f"wrong counts:\n{C_toy}"

counts = count_pairs(train, index)
assert counts.sum() == sum(len(s) + 1 for s in train), "the training captions give one pair per word plus one per caption"
assert counts[:, START].sum() == 0 and counts[END].sum() == 0, "nothing comes before <s> or after </s>"
print(f"{counts.sum()} pairs in {len(train)} training captions, {(counts > 0).sum()} of them distinct")
print("Part 1 passed")

## Part 2 (coding task): Add-k smoothing

Write `bigram_probs(counts, k, start_id)`, the table of $P(\text{next} \mid \text{current})$
with add-$k$ smoothing: add $k$ to the count of every token that can come next, then
divide each row by its new total. The start token never comes next, so its column stays 0.
With $k = 0$ the rows are the plain relative frequencies. The row of `</s>` is never used,
since nothing follows the end of a sentence, so whatever it holds is fine. Do not change
`counts`.

In [ ]:
def bigram_probs(counts, k, start_id):
    """Add-k smoothed P(next | current), with probability 0 for the start token."""
    # TODO
    raise NotImplementedError

In [ ]:
C4 = np.array([[0, 0, 3, 1], [0, 0, 0, 0], [0, 2, 0, 2], [0, 1, 1, 0]])
C4_before = C4.copy()
P4 = np.asarray(bigram_probs(C4, 0.5, 0))
assert P4.shape == (4, 4), f"expected a 4 x 4 table, got shape {P4.shape}"
assert np.allclose(P4[:, 0], 0), "the start token never comes next: its column should be 0"
assert np.allclose(P4[[0, 2, 3]].sum(axis=1), 1), f"each row should sum to 1, got sums {P4.sum(axis=1)}: divide by the row's smoothed total"
assert np.allclose(P4[0], [0, 0.5 / 5.5, 3.5 / 5.5, 1.5 / 5.5]), f"row of <s> with k = 0.5: expected (0, 0.5, 3.5, 1.5) / 5.5, got {P4[0]}"
assert np.allclose(P4[2], [0, 2.5 / 5.5, 0.5 / 5.5, 2.5 / 5.5]), f"row of a with k = 0.5: expected (0, 2.5, 0.5, 2.5) / 5.5, got {P4[2]}"
with np.errstate(invalid="ignore", divide="ignore"):
    P0 = np.asarray(bigram_probs(C4, 0, 0))
assert np.allclose(P0[0], [0, 0, 0.75, 0.25]), f"with k = 0 the rows are the relative frequencies, got {P0[0]}"
assert np.array_equal(C4, C4_before), "bigram_probs changed the counts it was given: work on a copy"

probs = bigram_probs(counts, 0.1, START)
assert np.allclose(np.delete(probs, END, axis=0).sum(axis=1), 1), "every row except that of </s> should sum to 1"
print("Part 2 passed")

## Part 3 (coding task): Perplexity, and choosing k

Write `perplexity(probs, sentences, index)`, the model's perplexity per token on a list of
sentences. Each sentence gives one prediction per word plus one for its `</s>`, each made
from the token before it, and the perplexity is $e$ to the mean of $-\ln P$ over all of
those predictions together. If the model gives some prediction probability 0, the
perplexity is infinite: return `inf`, not an error or `nan`.

The check then chooses $k$ by the validation captions and reports the test perplexity.

In [ ]:
def perplexity(probs, sentences, index):
    """e to the mean of -ln P(next | current) over every prediction, </s> included."""
    # TODO
    raise NotImplementedError

In [ ]:
toy_P = np.array([[0, 0, 0.5, 0.5],
                  [0, 1 / 3, 1 / 3, 1 / 3],
                  [0, 0.25, 0.25, 0.5],
                  [0, 0.5, 0.5, 0.0]])
pp = perplexity(toy_P, [["a"], ["b", "a"]], toy)
assert not np.isclose(pp, 2.0), "the prediction of </s> at the end of each sentence is missing"
assert not np.isclose(pp, (2 ** 1.5 + 2 ** (4 / 3)) / 2), "this is the mean of the two sentences' perplexities: pool all 5 predictions into one mean loss"
assert not np.isclose(pp, 2 ** ((1.5 + 4 / 3) / 2)), "this averages the two sentences' mean losses: pool all 5 predictions, so the longer sentence counts for more"
assert np.isclose(pp, 2 ** 1.4), f"7 bits of loss over 5 predictions: expected 2^1.4 = {2 ** 1.4:.4f}, got {pp}"
with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
    pp_zero = perplexity(toy_P, [["b", "b"]], toy)
assert pp_zero == np.inf, f"b never follows b in this table, so the perplexity should be inf, got {pp_zero}"
with np.errstate(divide="ignore", invalid="ignore", over="ignore", under="ignore"):
    pp_long = perplexity(toy_P, [["b", "a"]] * 3000, toy)
assert np.isfinite(pp_long) and np.isclose(pp_long, 2 ** (4 / 3)), f"3,000 copies of one sentence should give 2^(4/3) = {2 ** (4 / 3):.4f}, got {pp_long}: a product of 9,000 probabilities underflows to 0, so add up logs instead"

print("k         train    valid")
valid_ppl = {}
for k in [0, 0.001, 0.01, 0.03, 0.1, 0.3, 1]:
    with np.errstate(divide="ignore", invalid="ignore"):
        P = bigram_probs(counts, k, START)
        valid_ppl[k] = perplexity(P, valid, index)
        print(f"{k:<6} {perplexity(P, train, index):8.3f} {valid_ppl[k]:8.3f}")
best = min(valid_ppl, key=valid_ppl.get)
test_ppl = perplexity(bigram_probs(counts, best, START), test, index)
print(f"k chosen on validation: {best}. Test perplexity {test_ppl:.3f}, against {len(vocab) - 1} for a uniform guess")
assert valid_ppl[0] == np.inf, "with k = 0, a validation pair never seen in training has probability 0, so the perplexity should be inf"
assert best == 0.03 and abs(test_ppl - 3.226) < 0.002, f"expected k = 0.03 and a test perplexity of about {3.226}, got k = {best} and {test_ppl:.3f}"
print("Part 3 passed")

## Solution

### Part 1: Counting pairs

In [ ]:
def count_pairs(sentences, index):
    """V x V counts: entry [i, j] is how often token j comes right after token i."""
    C = np.zeros((len(index), len(index)), dtype=int)
    for s in sentences:
        ids = [index["<s>"]] + [index[w] for w in s] + [index["</s>"]]
        np.add.at(C, (ids[:-1], ids[1:]), 1)       # adds 1 for every pair, repeats included
    return C

**Why it works.** Each caption is wrapped and counted on its own, so no pair runs from
the end of one caption into the start of the next. `np.add.at` adds 1 for every pair in
the lists, repeats included. Plain indexing, `C[prev, nxt] += 1`, would add 1 only once to
a pair that occurs twice in the same call, a silent undercount.

### Part 2: Add-k smoothing

In [ ]:
def bigram_probs(counts, k, start_id):
    """Add-k smoothed P(next | current), with probability 0 for the start token."""
    smoothed = counts.astype(float) + k
    smoothed[:, start_id] = 0.0                    # <s> is never a next token
    with np.errstate(invalid="ignore"):            # with k = 0 the unused row of </s> is 0 / 0
        return smoothed / smoothed.sum(axis=1, keepdims=True)

**Why it works.** Adding $k$ to the $V - 1$ tokens that can come next adds $k(V - 1)$ to
a row's total, and dividing by the new total makes the row sum to 1 again. Adding $k$ to
all $V$ columns and dividing by that larger total before zeroing the start column would
leave every row short of 1, and
adding it to all $V$ without zeroing would keep a share of probability for a token that
never occurs. A pair never seen in training now gets $k$ over its row's total instead of 0.

### Part 3: Perplexity, and choosing k

In [ ]:
def perplexity(probs, sentences, index):
    """e to the mean of -ln P(next | current) over every prediction, </s> included."""
    prev, nxt = [], []
    for s in sentences:
        ids = [index["<s>"]] + [index[w] for w in s] + [index["</s>"]]
        prev += ids[:-1]
        nxt += ids[1:]
    with np.errstate(divide="ignore"):             # log(0) = -inf gives an infinite perplexity
        log_p = np.log(probs[prev, nxt])
    return float(np.exp(-log_p.mean()))

**Why it works.** Collecting every (current, next) pair first and taking one mean of the
logs pools all predictions, so a long caption counts for more than a short one, and no
product of thousands of probabilities is ever formed. A probability of 0 gives a log of
$-\infty$, so the mean is $-\infty$ and the perplexity $\infty$, the honest answer for a
model that rules out something the text contains.

The table shows why $k$ is chosen on validation captions. The training perplexity is
lowest at $k = 0$, because every training pair was seen and smoothing only moves
probability away from them. On the validation captions, $k = 0$ is infinite, because some
of their pairs never occur in the 60 training captions, and $k = 1$ spreads so much
probability over unseen pairs that the perplexity nearly doubles. The validation minimum, at
$k = 0.03$, is then measured once on the test captions: about 3.2, against 28 for a model
that guesses uniformly.